# 03 Predict Matchup

Choose two teams and a series format, then estimate win probabilities from Elo plus recent form.

In [17]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict

import pandas as pd

from moirai.models.backtest import calibrate_form_weights
from moirai.pipeline.store import load_matches
from moirai.predict import FormWeights, predict_matchup

TEAM_A = "Top Esports"
TEAM_B = "Team WE"
BEST_OF = 5

matches = load_matches()
calibration = calibrate_form_weights(matches, min_prior_games=3, epochs=3000, learning_rate=0.03, l2=0.01)
calibration_improvement = calibration.baseline_log_loss - calibration.adjusted_log_loss
form_weights = calibration.weights if calibration_improvement > 0 else FormWeights()

prediction = predict_matchup(
    matches,
    TEAM_A,
    TEAM_B,
    best_of=BEST_OF,
    form_weights=form_weights,
)

summary = asdict(prediction)
summary.pop("features")
summary.pop("feature_adjustments")
pd.Series(summary)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


team_a                                Top Esports
team_b                                    Team WE
best_of                                         5
team_a_rating                         1673.231928
team_b_rating                         1549.510202
team_a_game_probability                  0.670887
team_a_series_probability                0.796336
team_b_series_probability                0.203664
team_a_adjusted_game_probability         0.675604
team_a_adjusted_series_probability       0.803185
team_b_adjusted_series_probability       0.196815
dtype: object

In [18]:
pd.Series(
    {
        "backtest_rows": calibration.rows_used,
        "train_rows": calibration.train_rows,
        "test_rows": calibration.test_rows,
        "baseline_log_loss": calibration.baseline_log_loss,
        "adjusted_log_loss": calibration.adjusted_log_loss,
        "log_loss_improvement": calibration_improvement,
        "baseline_accuracy": calibration.baseline_accuracy,
        "adjusted_accuracy": calibration.adjusted_accuracy,
        "learned_recent_form_weight": calibration.weights.recent_form,
        "learned_streak_weight": calibration.weights.streak,
        "learned_rest_weight": calibration.weights.rest,
        "using_calibrated_weights": calibration_improvement > 0,
    }
)

backtest_rows                     4289
train_rows                        3002
test_rows                         1287
baseline_log_loss             0.638889
adjusted_log_loss             0.638618
log_loss_improvement          0.000271
baseline_accuracy             0.635587
adjusted_accuracy             0.636364
learned_recent_form_weight   -0.051292
learned_streak_weight        -0.015219
learned_rest_weight          -0.019251
using_calibrated_weights          True
dtype: object

In [19]:
pd.concat(
    [
        pd.Series(prediction.features, name="value"),
        pd.Series(prediction.feature_adjustments, name="value"),
    ]
)

team_a_recent_games            10.000000
team_b_recent_games            10.000000
team_a_recent_win_rate          0.600000
team_b_recent_win_rate          0.800000
recent_win_rate_diff           -0.200000
team_a_current_streak           3.000000
team_b_current_streak           5.000000
streak_diff                    -2.000000
team_a_days_since_last_game     6.000000
team_b_days_since_last_game     5.000000
recent_form_logit               0.010258
streak_logit                    0.030437
rest_logit                     -0.019251
Name: value, dtype: float64

In [16]:
# Kelly staking calculator for binary markets such as Polymarket.
# Use your model probability for the team and the market-implied probability/price.
# This is a sizing formula, not a guarantee of profit.

BANKROLL = 1417
TEAM = TEAM_A
# MODEL_PROBABILITY = prediction.team_a_adjusted_series_probability
MODEL_PROBABILITY = prediction.team_a_game_probability
MARKET_PROBABILITY = 0.52  # e.g. a Polymarket YES price of 50c implies 0.50
CUSTOM_KELLY_MULTIPLIER = 0.1


def kelly_fraction(model_probability: float, market_probability: float) -> float:
    if not 0 < market_probability < 1:
        raise ValueError("market_probability must be between 0 and 1")
    if not 0 <= model_probability <= 1:
        raise ValueError("model_probability must be between 0 and 1")

    decimal_odds = 1 / market_probability
    net_odds = decimal_odds - 1
    fraction = (model_probability * decimal_odds - 1) / net_odds
    return max(0, fraction)


full_kelly = kelly_fraction(MODEL_PROBABILITY, MARKET_PROBABILITY)
kelly_table = pd.DataFrame(
    [
        {"strategy": "full_kelly", "kelly_multiplier": 1.0},
        {"strategy": "half_kelly", "kelly_multiplier": 0.5},
        {"strategy": "quarter_kelly", "kelly_multiplier": 0.25},
        {"strategy": "custom_kelly", "kelly_multiplier": CUSTOM_KELLY_MULTIPLIER},
    ]
)
kelly_table["team"] = TEAM
kelly_table["bankroll"] = BANKROLL
kelly_table["model_probability"] = MODEL_PROBABILITY
kelly_table["market_probability"] = MARKET_PROBABILITY
kelly_table["edge"] = MODEL_PROBABILITY - MARKET_PROBABILITY
kelly_table["full_kelly_fraction"] = full_kelly
kelly_table["bet_fraction"] = kelly_table["full_kelly_fraction"] * kelly_table["kelly_multiplier"]
kelly_table["bet_amount"] = kelly_table["bankroll"] * kelly_table["bet_fraction"]
kelly_table

,strategy,kelly_multiplier,team,bankroll,model_probability,market_probability,edge,full_kelly_fraction,bet_fraction,bet_amount
0,full_kelly,1.00,Dplus KIA,1417,0.617635,0.52,0.097635,0.203407,0.203407,288.227690
1,half_kelly,0.50,Dplus KIA,1417,0.617635,0.52,0.097635,0.203407,0.101703,144.113845
2,quarter_kelly,0.25,Dplus KIA,1417,0.617635,0.52,0.097635,0.203407,0.050852,72.056922
3,custom_kelly,0.10,Dplus KIA,1417,0.617635,0.52,0.097635,0.203407,0.020341,28.822769
